# CEM vs ACEM (Shared Critic) — Leakage Analysis

Compares CEM and ACEM across datasets using:
- Task / concept accuracies
- OIS / NIS
- MI leakage scores (CTL, ICL on mix embeddings)
- Fidelity & Leakage probes (linear + MLP)
- Spectral task-leakage
- Inter-concept entanglement matrix
- Canonical Correlation Analysis (CCA)

In [ ]:
import joblib
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

os.chdir('/Users/ahabisaac/Projects/xai-concept-leakage')

PALETTE = {'CEM': '#9467bd', 'ACEM': '#17becf'}
SAVE_DIR = 'saved_charts/cem_acem'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
RESULT_FILES = {
    ('TabularToy', 'CEM'):  'results/results_tabulartoy_cem.dict',
    ('TabularToy', 'ACEM'): 'results/results_tabulartoy_acem.dict',
    ('dSprites',   'CEM'):  'results/results_dsprites_cem.dict',
    ('dSprites',   'ACEM'): 'results/results_dsprites_acem.dict',
    ('CUB',        'CEM'):  'results/results_cub_cem.dict',   # CEM and ACEM in same file
}

raw = {}
for key, path in RESULT_FILES.items():
    if os.path.exists(path):
        raw[key] = joblib.load(path)
        print(f'Loaded {key}: {len(raw[key])} checkpoints')
    else:
        print(f'Missing: {path}')

In [ ]:
def get_model_type(checkpoint_name):
    if 'CRCEM' in checkpoint_name or 'shared_critic' in checkpoint_name:
        return 'ACEM'
    return 'CEM'

def get_lam_c(checkpoint_name):
    m = re.search(r'lam_c([\d\.]+)', checkpoint_name)
    return float(m.group(1)) if m else None

def get_fold(checkpoint_name):
    m = re.search(r'fold_(\d+)', checkpoint_name)
    return int(m.group(1)) if m else None

SCALAR_KEYS = [
    # Accuracies
    ('test', 'y_accuracy'),
    ('test', 'c_accuracy'),
    # OIS / NIS
    ('test', 'ois'),
    ('test', 'nis'),
    # MI leakage
    ('test', 'CT_MI_mix'),
    ('test', 'IC_MI_mix'),
    ('test', 'CT_MI_adjusted_mix'),
    # Linear probes
    ('test', 'linear_fidelity_mean'),
    ('test', 'linear_leakage_mean'),
    ('test', 'linear_probe_y_avg'),
    ('test', 'linear_probe_c_others_avg'),
    # MLP probes
    ('test', 'mlp_fidelity_mean'),
    ('test', 'mlp_leakage_mean'),
    # Entanglement
    ('test', 'entanglement_linear_diag_mean'),
    ('test', 'entanglement_linear_offdiag_mean'),
    ('test', 'entanglement_mlp_diag_mean'),
    ('test', 'entanglement_mlp_offdiag_mean'),
    # CCA
    ('test', 'cca_mean_abs'),
]

def extract_scalar(result, split, key):
    val = result.get(split, {}).get(key)
    if val is None:
        return np.nan
    if isinstance(val, (list, np.ndarray)):
        return float(np.mean(val))
    return float(val)

def results_to_df(raw_dict, dataset, model_type):
    rows = []
    for cp_name, result in raw_dict.items():
        row = {
            'dataset': dataset,
            'model': get_model_type(cp_name) if model_type == 'auto' else model_type,
            'lam_c': get_lam_c(cp_name),
            'fold': get_fold(cp_name),
            'checkpoint': cp_name,
        }
        for split, key in SCALAR_KEYS:
            row[key] = extract_scalar(result, split, key)
        # spectral curves stored separately
        row['_spectral_accs'] = result.get('test', {}).get('spectral_accs_mean_per_k')
        row['_spectral_varexp'] = result.get('test', {}).get('spectral_varexp_mean_per_k')
        row['_entangle_linear'] = result.get('test', {}).get('entanglement_linear')
        row['_entangle_mlp'] = result.get('test', {}).get('entanglement_mlp')
        rows.append(row)
    return pd.DataFrame(rows)

dfs = []
for (dataset, model_type), raw_dict in raw.items():
    # CUB file contains both CEM and ACEM — auto-detect
    auto = model_type if dataset != 'CUB' else 'auto'
    dfs.append(results_to_df(raw_dict, dataset, auto))

df = pd.concat(dfs, ignore_index=True)
print(df.groupby(['dataset', 'model', 'lam_c']).size().to_string())

In [ ]:
BAR_METRICS = [
    ('y_accuracy',                    'Task Accuracy'),
    ('c_accuracy',                    'Concept Accuracy'),
    ('CT_MI_adjusted_mix',            'CTL (adjusted, mix)'),
    ('IC_MI_mix',                     'ICL (mix)'),
    ('ois',                           'OIS'),
    ('nis',                           'NIS'),
    ('linear_fidelity_mean',          'Linear Fidelity (e_i → c_i)'),
    ('linear_leakage_mean',           'Linear Leakage (e_i → y)'),
    ('mlp_fidelity_mean',             'MLP Fidelity (e_i → c_i)'),
    ('mlp_leakage_mean',              'MLP Leakage (e_i → y)'),
    ('entanglement_linear_diag_mean', 'Entanglement Diag (linear)'),
    ('entanglement_linear_offdiag_mean', 'Entanglement Off-diag (linear)'),
    ('entanglement_mlp_diag_mean',    'Entanglement Diag (MLP)'),
    ('entanglement_mlp_offdiag_mean', 'Entanglement Off-diag (MLP)'),
    ('cca_mean_abs',                  'CCA Mean |Correlation|'),
]

datasets = df['dataset'].unique()

for dataset in datasets:
    sub = df[df['dataset'] == dataset].copy()
    n_metrics = len(BAR_METRICS)
    ncols = 3
    nrows = (n_metrics + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
    axes = axes.flatten()

    for ax, (col, title) in zip(axes, BAR_METRICS):
        plot_sub = sub.dropna(subset=[col])
        if plot_sub.empty:
            ax.set_visible(False)
            continue
        sns.barplot(
            data=plot_sub, x='lam_c', y=col, hue='model',
            hue_order=['CEM', 'ACEM'], palette=PALETTE,
            errorbar='sd', capsize=0.1, ax=ax
        )
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel(r'$\lambda_c$')
        ax.set_ylabel('')
        ax.legend(title='Model', fontsize=9)

    for ax in axes[n_metrics:]:
        ax.set_visible(False)

    fig.suptitle(f'{dataset}: CEM vs ACEM', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/{dataset}_bar_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {dataset}')

In [ ]:
# Spectral task-leakage: accuracy vs number of PCA components
# One panel per dataset, CEM vs ACEM lines

for dataset in datasets:
    sub = df[df['dataset'] == dataset]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, model in zip(axes, ['CEM', 'ACEM']):
        rows = sub[sub['model'] == model].dropna(subset=['_spectral_accs'])
        if rows.empty:
            ax.set_visible(False)
            continue

        for lam_c, grp in rows.groupby('lam_c'):
            curves = np.array([r for r in grp['_spectral_accs'] if r is not None])
            varexp = np.array([r for r in grp['_spectral_varexp'] if r is not None])
            if curves.size == 0:
                continue
            mean_acc = curves.mean(axis=0)
            mean_var = varexp.mean(axis=0)
            ks = np.arange(1, len(mean_acc) + 1)

            color = plt.cm.viridis(lam_c / (rows['lam_c'].max() + 1e-6))
            ax.plot(ks, mean_acc, label=f'λ_c={lam_c}', color=color)
            ax2 = ax.twinx()
            ax2.plot(ks, mean_var, linestyle='--', color=color, alpha=0.4)
            ax2.set_ylabel('Cumul. Var. Explained', fontsize=9, color='grey')
            ax2.tick_params(axis='y', labelcolor='grey')

        ax.set_xlabel('PCA components (k)')
        ax.set_ylabel('Acc predicting y from e_i top-k PCs')
        ax.set_title(f'{dataset} — {model}', fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.3)

    plt.suptitle(f'Spectral Task-Leakage: {dataset}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/{dataset}_spectral.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Entanglement matrix heatmaps — one representative checkpoint per (dataset, model, lam_c)
# Uses the mean matrix across folds

def mean_matrix(grp, col):
    mats = [np.array(m) for m in grp[col] if m is not None]
    if not mats:
        return None
    return np.mean(mats, axis=0)

for dataset in datasets:
    sub = df[df['dataset'] == dataset]
    lam_cs = sorted(sub['lam_c'].dropna().unique())
    models = ['CEM', 'ACEM']

    # 2 probe types × n_models × n_lam_c
    for probe, col in [('Linear', '_entangle_linear'), ('MLP', '_entangle_mlp')]:
        ncols = len(lam_cs)
        fig, axes = plt.subplots(len(models), ncols,
                                  figsize=(4 * ncols, 4 * len(models)),
                                  squeeze=False)
        for r, model in enumerate(models):
            for c, lam_c in enumerate(lam_cs):
                ax = axes[r][c]
                grp = sub[(sub['model'] == model) & (sub['lam_c'] == lam_c)]
                mat = mean_matrix(grp, col)
                if mat is None:
                    ax.set_visible(False)
                    continue
                sns.heatmap(mat, ax=ax, vmin=0, vmax=1, cmap='YlOrRd',
                            annot=mat.shape[0] <= 10, fmt='.2f',
                            cbar=(c == ncols - 1),
                            xticklabels=False, yticklabels=False)
                ax.set_title(f'{model}  λ_c={lam_c}', fontsize=10)
                if c == 0:
                    ax.set_ylabel(model, fontsize=11, fontweight='bold')

        fig.suptitle(f'{dataset}: {probe} Entanglement Matrix (e_i → c_j)',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{SAVE_DIR}/{dataset}_entanglement_{probe.lower()}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# Summary table: mean ± std across folds and lam_c values for each (dataset, model)

summary_cols = [
    'y_accuracy', 'c_accuracy', 'ois', 'nis',
    'CT_MI_adjusted_mix', 'IC_MI_mix',
    'linear_fidelity_mean', 'linear_leakage_mean',
    'mlp_fidelity_mean', 'mlp_leakage_mean',
    'entanglement_linear_diag_mean', 'entanglement_linear_offdiag_mean',
    'cca_mean_abs',
]

summary = (
    df.groupby(['dataset', 'model', 'lam_c'])[summary_cols]
    .agg(['mean', 'std'])
    .round(3)
)
summary